# De-immunizing protein designs with MHC class II–aware ProteinMPNN

Redesign a protein sequence for a fixed backbone so that its predicted **MHC class II
(T-helper) epitope burden** is reduced, while the backbone (and, for a binder, its target)
is held fixed. This is the sequence-generation step from *"Reducing Predicted MHC Class II
Epitope Burden in De Novo Protein Design"* — the de-immunization lives entirely in the
model weights, so you run stock [ProteinMPNN](https://github.com/dauparas/ProteinMPNN)
with the released fine-tuned checkpoint.

**What this notebook does:** clone ProteinMPNN → download the released weights → give it a
PDB (by ID or upload) and the chain(s) to redesign → generate sequences with the
de-immunized model and, for comparison, base ProteinMPNN → download the results →
*optionally* validate a design with **Boltz-2** (does it still fold / still bind its target?).

To *quantify* the epitope-burden reduction on the generated sequences, use `score_burden.py`
in the code repository (it needs the MHC-II PWM data bundle); this notebook covers generation
and structure/binding validation.

> Runtime: **Runtime → Change runtime type → GPU** is recommended but not required
> (ProteinMPNN is small and runs on CPU too).

## 1. Setup

In [ ]:
# Clone stock ProteinMPNN (unmodified) and check the runtime.
import torch, os, subprocess, sys
print('GPU available:', torch.cuda.is_available(),
      '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'))
if not os.path.isdir('/content/ProteinMPNN'):
    subprocess.run(['git','clone','-q','https://github.com/dauparas/ProteinMPNN',
                    '/content/ProteinMPNN'], check=True)
PMPNN = '/content/ProteinMPNN/protein_mpnn_run.py'
print('ProteinMPNN ready:', os.path.isfile(PMPNN))

## 2. Download the de-immunized weights

The released checkpoints are hosted on Zenodo. **Set the record id below** (available from
the paper's Data & Code Availability section once deposited). The base weights
(`v_48_020`) ship with the ProteinMPNN clone, so only the de-immunized checkpoint is fetched.

In [ ]:
ZENODO_RECORD = ''  #@param {type:'string'}
# deimmu_deployed.pt is THE model the paper reports (41.4% predicted class II
# reduction). deimmu_mhc2only.pt drops the class I objective. The cape_creg*/
# base_lambda0 files are the earlier lower-learning-rate sweep arms, kept for
# reproducing the charge-regulariser sweep -- not recommended defaults.
CAPE_WEIGHT   = 'deimmu_deployed.pt'  #@param ['deimmu_deployed.pt','deimmu_mhc2only.pt','cape_creg1.0.pt','cape_creg0.5.pt','base_lambda0.pt']

import os, urllib.request
os.makedirs('/content/weights', exist_ok=True)
assert ZENODO_RECORD.strip(), 'Set ZENODO_RECORD to the deposited record id first.'
url = f'https://zenodo.org/records/{ZENODO_RECORD}/files/{CAPE_WEIGHT}?download=1'
dst = f'/content/weights/{CAPE_WEIGHT}'
if not os.path.isfile(dst):
    print('downloading', url)
    urllib.request.urlretrieve(url, dst)
CAPE_MODEL = CAPE_WEIGHT[:-3]  # ProteinMPNN --model_name is the filename without .pt
print('cape weights:', dst, '| model_name:', CAPE_MODEL)

## 3. Input structure and redesign settings

Provide a **PDB ID** (fetched from RCSB) or upload your own file. Give the chain(s) to
**redesign** (e.g. `A`, or `AC` for two); every other chain is held fixed as context, which
is how the target of a binder is preserved. For a monomer, give its single chain id.

In [ ]:
PDB_ID     = '' #@param {type:'string'}
UPLOAD     = False #@param {type:'boolean'}
REDESIGN_CHAINS = 'A' #@param {type:'string'}
NUM_SEQUENCES   = 8 #@param {type:'integer'}
SAMPLING_TEMP   = 0.1 #@param {type:'number'}
SEED            = 37 #@param {type:'integer'}
ALSO_RUN_BASE   = True #@param {type:'boolean'}

import os, urllib.request
os.makedirs('/content/input', exist_ok=True)
if UPLOAD:
    from google.colab import files
    up = files.upload()
    RAW = '/content/input/' + list(up.keys())[0]
else:
    assert PDB_ID.strip(), 'Give a PDB_ID or set UPLOAD=True.'
    RAW = f'/content/input/{PDB_ID.strip().lower()}.pdb'
    urllib.request.urlretrieve(f'https://files.rcsb.org/download/{PDB_ID.strip().upper()}.pdb', RAW)
print('input structure:', RAW)

## 4. Clean the structure

Keep only the chains involved (the redesigned chain plus any target chains you name below),
standard amino acids, altloc A, and the first model — the same minimal-complex preparation
used in the paper. List **all** chains to keep (redesigned + fixed target); leave blank to
keep every chain in the file.

In [ ]:
KEEP_CHAINS = '' #@param {type:'string'}  # e.g. 'A B' ; blank = keep all
AA3 = set('ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL MSE'.split())
keep = set(KEEP_CHAINS.split()) if KEEP_CHAINS.strip() else None
CLEAN = '/content/input/clean.pdb'
n=0; per={}
with open(RAW) as f, open(CLEAN,'w') as o:
    for line in f:
        if line.startswith('ENDMDL'): break
        if line.startswith('TER'):
            o.write(line); continue
        if line[:6].strip() in ('ATOM','HETATM'):
            if line[17:20].strip() not in AA3: continue
            if line[16] not in (' ','A'): continue
            ch = line[21]
            if keep is not None and ch not in keep: continue
            o.write(line); n+=1; per[ch]=per.get(ch,0)+1
    o.write('END\n')
print('kept', n, 'atoms; per-chain:', per)
assert all(c in per for c in REDESIGN_CHAINS), \
    f'redesign chain(s) {REDESIGN_CHAINS} not found among {sorted(per)}'

## 5. Generate sequences

In [ ]:
import subprocess, os
def run(model_name, weights_dir, tag):
    out = f'/content/out_{tag}'
    cmd = ['python', PMPNN, '--pdb_path', CLEAN,
           '--pdb_path_chains', ' '.join(list(REDESIGN_CHAINS)),
           '--model_name', model_name,
           '--out_folder', out,
           '--num_seq_per_target', str(NUM_SEQUENCES),
           '--sampling_temp', str(SAMPLING_TEMP),
           '--seed', str(SEED), '--batch_size', '1']
    if weights_dir: cmd[3:3] = ['--path_to_model_weights', weights_dir]
    print('running', tag, '...')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:]); raise RuntimeError(tag+' failed')
    return out

cape_out = run(CAPE_MODEL, '/content/weights', 'cape')
base_out = run('v_48_020', '', 'base') if ALSO_RUN_BASE else None
print('done')

## 6. Results

ProteinMPNN writes a FASTA per run (first record = native, then the designs). Below prints
the designs and zips both runs for download.

In [ ]:
import glob, os, shutil
def show(out, tag):
    fa = glob.glob(os.path.join(out,'seqs','*.fa'))
    if not fa: print('no output for', tag); return
    txt = open(fa[0]).read()
    recs = txt.split('>')[1:]
    print(f'=== {tag}: {len(recs)-1} designs (first is native) ===')
    for r in recs[1:]:
        header, *seq = r.strip().split('\n')
        print('>'+header.split(',')[0]); print(''.join(seq))
    print()
show(cape_out, 'de-immunized (CAPE)')
if base_out: show(base_out, 'base ProteinMPNN')

shutil.make_archive('/content/deimmunize_results','zip','/content', 'out_cape')
if base_out: shutil.make_archive('/content/base_results','zip','/content','out_base')
from google.colab import files as _f
_f.download('/content/deimmunize_results.zip')

## 7. Validate structure / binding with Boltz-2 (optional)

Fold a redesigned sequence with [Boltz-2](https://github.com/jwohlwend/boltz) to check the
design still folds (monomer) or still binds its target (complex). For a complex this follows
the paper's protocol: the **natural target is folded with an MMseqs2-server MSA** and the
**de novo binder single-sequence**, and the interface **ipTM** is the readout (for a monomer,
pTM / pLDDT are reported instead).

> **This step is heavy.** Boltz-2 downloads multi-GB model weights on first run, needs a GPU,
> and takes minutes per prediction, so it defaults to folding a single design. Large complexes
> may exceed free-Colab GPU memory.

In [ ]:
# Install Boltz-2 (model weights download on first predict; GPU strongly recommended).
import subprocess
subprocess.run(['pip','install','-q','boltz'], check=True)
print('boltz installed:', subprocess.run(['boltz','--help'],capture_output=True,text=True).returncode==0)

In [ ]:
FOLD_MODEL = 'cape' #@param ['cape','base','both']
N_DESIGNS_TO_FOLD = 1 #@param {type:'integer'}

import os, glob, json, subprocess
T2O = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G',
       'HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S',
       'THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}

def chain_seqs(pdb):
    out={}
    for line in open(pdb):
        if line.startswith('ATOM') and line[12:16].strip()=='CA':
            out.setdefault(line[21], []).append(T2O.get(line[17:20].strip(),'X'))
    return {c:''.join(s) for c,s in out.items()}

cseq   = chain_seqs(CLEAN)
binder = list(REDESIGN_CHAINS)
targets = [c for c in cseq if c not in binder]
is_complex = len(targets) > 0
print(('complex' if is_complex else 'monomer'), '| target chains:', targets, '| binder:', binder)

def designs_of(out):
    fa = glob.glob(os.path.join(out,'seqs','*.fa'))[0]
    recs = open(fa).read().split('>')[1:]
    return [''.join(r.strip().split('\n')[1:]).replace('/','') for r in recs[1:]]

def make_yaml(design_seq, path):
    lines = ['version: 1', 'sequences:']
    for c in targets:                       # natural target(s): MSA via server
        lines += ['  - protein:', f'      id: {c}', f'      sequence: {cseq[c]}']
    bid = binder[0]                          # binder / monomer design: single-sequence
    lines += ['  - protein:', f'      id: {bid}', f'      sequence: {design_seq}', '      msa: empty']
    open(path,'w').write('\n'.join(lines)+'\n')

def run_boltz(yaml_path, odir):
    cmd = ['boltz','predict',yaml_path,'--out_dir',odir,'--override']
    if is_complex: cmd.append('--use_msa_server')   # MSA for target chains; binder is msa:empty
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:]); raise RuntimeError('boltz predict failed')

def confidence(odir):
    j = glob.glob(os.path.join(odir,'**','confidence*.json'), recursive=True)
    return json.load(open(j[0])) if j else {}

runs = []
if FOLD_MODEL in ('cape','both'): runs.append(('cape', cape_out))
if FOLD_MODEL in ('base','both') and base_out: runs.append(('base', base_out))

os.makedirs('/content/boltz', exist_ok=True)
rows = []
for tag, out in runs:
    for i, seq in enumerate(designs_of(out)[:max(1, N_DESIGNS_TO_FOLD)]):
        name = f'{tag}_{i}'
        make_yaml(seq, f'/content/boltz/{name}.yaml')
        print('folding', name, '...')
        run_boltz(f'/content/boltz/{name}.yaml', f'/content/boltz/{name}')
        d = confidence(f'/content/boltz/{name}')
        rows.append((name, d.get('iptm'), d.get('ptm'), d.get('complex_plddt')))

def fmt(x): return f'{x:.3f}' if isinstance(x,(int,float)) else '  -  '
print('\n design       ipTM    pTM    pLDDT')
for name, iptm, ptm, plddt in rows:
    print(f' {name:11s} {fmt(iptm)}  {fmt(ptm)}  {fmt(plddt)}')
print('\nipTM = interface confidence (binder<->target); compare cape vs base.' if is_complex
      else '\npTM/pLDDT = foldability; for self-consistency TM vs native see the repo.')

## Next steps

- **Quantify the reduction.** Score predicted MHC-II burden on the base vs de-immunized
  FASTAs with `application/score_burden.py` in the code repository (needs the PWM bundle).
- **Second binding validator.** AF2 initial-guess (a methodologically independent check) is
  in `application/af2_initial_guess/` in the repository.

Repository: `[repository URL to be finalized]` · Paper: `[DOI to be finalized]`.
Built on ProteinMPNN (Dauparas et al. 2022), CAPE-MPNN (Gasser et al. 2025),
and Boltz-2 (Passaro et al. 2025).